## **What data structure to use for iterating over the variants?**

### **First implementation** (Single Chromosome)
The first implementation was to use a queue that stored both the variant info, as well as the features to be used in calculating the statistics. 

The problem with this approach was that if we were to process multiple BAM files, we would need create the list of variants over again as we are popping variants once they are processed from the front of the queue. The approach for this would be to separate the variant info into its own vector; then for each variant, create a struct that has a lifetime non mutable reference to the variant info, and another field with the features. Lifetime notation is needed as the compiler needs to know beforehand which fields. [Need to further understand lifetimes.]

However, since multiple BAM processing is not necessary, we can instead keep it as it is currently.

```mermaid
flowchart TD
    subgraph Implementation if multi-bam processing is needed

    var_vector_1@{ shape: bow-rect, label: "Vector of variant_info
    [Vector]
    chrom: String,
	pos: u64,
	refr: String,
	alt: String,
    vartype: VarType," }

    var_vector_1 --> var_queue_2@{ shape: bow-rect, label: "Variants Vector
    [Vector Queue (vecdeque)]
    variant: &variant_info
    features: LocusFeatures," }

    end

    subgraph Current implementation

    var_queue_1@{ shape: bow-rect, label: "Variants Queue
    [Vector Queue (vecdeque)]
    chrom: String,
	pos: u64,
	refr: String,
	alt: String,
    vartype: VarType,
    features: LocusFeatures," }

    end 
```

### **Second implementation attempt** (Multi-chromosome)
The first approach only works if all the variants are from a single chromosome. Since we are no longer using the variant pileup approach, and instead iterating over reads and seeing if they overlap with the variants, we need to instead call `fetch` in the bam reader. `fetch` is only able to fetch reads within a given position range for a single chromosome, therefore we needed to somehow separate the variants so that we can call `fetch` a subset.

With the `VecDeque` already, one thing we could do would be to calculate the number of variants for each chromosome and save this in a vector; then we could subset the queue by [0..number of variants], processing the chromosome subset which will be popped once completed. Then we repeat this for the rest of the chromosomes, since the start of the queue will be the start of a new chromosome with the range ending in the next number of variants. However, since VecDeque is implemented as a ring buffer, the elements inside may not be contiguous if they wrap wround the end of the physical buffer and so we cannot just simply slice a subset (Can do if we use `as_slice` or `make_contiguous` but unsure about performance/feasability). 

We could use either `drain` or `split_off`: 
- `drain` will remove a given range from the queue and return them as an iterator; however, since we do not want to consume the variant after iterating over it as it could align to another read, we would have to add the trait implementation `peekable` which has methods that can look at the next element in the iterator without consuming it. Downside of this approach is that the remaining elements of the queue will need to be shifted to the front of the queue [Time complexity: O(M) where M is the remaining variants in the queue | Space complexity: O(1) as we do not need to any new allocations].
- `split_off` will split the given range into a new `VecDeque`. The problem with this is that there is O(N) time complexity, where N is the all the elements in the queue, as we need to allocate the split to a new chunk of memory, while also shifting the remaining elements. There is also O(S) space complexity, where S is the number of split elements.

Another approach would be to separate the single `VecDeque` into separate queues for each chromosome when initializing it. For example, if the variants list contains chromosomes 1, 2, 3 and 4, then we would create a structure for each of these containing the chromosome and a `VecDeque`. This would essentially create a bucket of variants for each chromosome, which we could then append to a vector; as we are assuming the variants are sorted by at least chromosome and position, we do not have to do any sorting. I have implemented this approach as it is the easiest to understand without the complexity of splitting the queue if we want to implement multithreading later.

```mermaid
flowchart TD
    subgraph Current implementation

    var_queue_1@{ shape: bow-rect, label: "Variants Queue
    [VecDeque]
    chrom: String,
	pos: u64,
	refr: String,
	alt: String,
    vartype: VarType,
    features: LocusFeatures," }

    var_queue_1 --> bucket@{ shape: docs, label: "Chromosome buckets 
    [Struct]
    chrom: String,
    variants: Variants Queue" }

    bucket --> parse@{ shape: bow-rect, label: "Parsed variants
    [Vec]
    chroms: Chromosome bucket" }

    end 
```
